In [1]:
#!pip install -q --upgrade keras-hub
#!pip install -q --upgrade keras  # Upgrade to Keras 3.
#!pip install --upgrade keras-nlp
!pip install keras==3.5.0 keras-hub==0.18.1 keras-nlp==0.18.1 tensorflow==2.18.0

import os
import keras_hub
import keras
import tensorflow as tf
import tensorflow.data as tf_data
import tensorflow.strings as tf_strings

import os
import sqlite3
import pandas as pd
import kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.9 MB/s eta 0:00:00
  Attempting uninstall: keras
    Found existing installation: keras 3.8.0
    Uninstalling keras-3.8.0:
      Successfully uninstalled keras-3.8.0


In [2]:
BATCH_SIZE = 64
MIN_STRING_LEN = 512
SEQ_LEN = 128

EMBED_DIM = 256
FEED_FORWARD_DIM = 128
NUM_HEADS = 3
NUM_LAYERS = 2
VOCAB_SIZE = 1000

EPOCHS = 14

NUM_TOKENS_TO_GENERATE = 80

In [3]:
path = kagglehub.dataset_download("dhruvildave/wikibooks-dataset")

print("Path to dataset files:", path)
print(os.listdir(path))

conn = sqlite3.connect(f'{path}/wikibooks.sqlite')

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)

df = pd.read_sql("SELECT * FROM ru;", conn)
print(df)

#text = " ".join((df['abstract'].fillna('') + " " + df['body_text'].fillna('')).tolist())
#text = " ".join(df['abstract'].fillna('').tolist())

#text = df['abstract'].fillna('').tolist()

text = (df['abstract'].fillna('') + " " + df['body_text'].fillna('')).str.slice(0, 2000).tolist()

split_idx = int(0.9 * len(text))
train_lines = text[:split_idx]
val_lines = text[split_idx:]

os.makedirs("wikibooks_tmp", exist_ok=True)
with open("wikibooks_tmp/train.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(train_lines))
with open("wikibooks_tmp/valid.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(val_lines))

raw_train_ds = (
    tf_data.TextLineDataset("wikibooks_tmp/train.txt")
    .filter(lambda x: tf_strings.length(x) > MIN_STRING_LEN)
    .shuffle(buffer_size=256)
    .batch(BATCH_SIZE)
)

raw_train_ds = raw_train_ds.take(8000)

raw_val_ds = (
    tf_data.TextLineDataset("wikibooks_tmp/valid.txt")
    .filter(lambda x: tf_strings.length(x) > MIN_STRING_LEN)
    .batch(BATCH_SIZE)
)

100%|██████████| 1.82G/1.82G [01:26<00:00, 22.7MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/dhruvildave/wikibooks-dataset/versions/7
['wikibooks.sqlite']
   name
0    pl
1    hu
2    he
3    nl
4    ja
5    ru
6    it
7    en
8    es
9    pt
10   de
11   fr
                                                  title  \
0     Викиучебник: Техника и технология средств масс...   
1              Викиучебник: АОН/Пилотское свидетельство   
2     Викиучебник: Книга программиста/Структуры данн...   
3     Викиучебник: Тесты НМО/Гигиенические основы и ...   
4                      Викиучебник: Коктейли/Пенная фея   
...                                                 ...   
8208                    Викиучебник: Коктейли/Чики-пуки   
8209                 Викиучебник: Коктейли/Будьте добры   
8210             Викиучебник: Коктейли/Античный дайкири   
8211  Викиучебник: Открытый учебник нейронаук/Патоло...   
8212  Викиучебник: Тесты НМО/Аллергический конъюнктивит   

                                                    url  \
0     https

In [4]:
vocab = keras_hub.tokenizers.compute_word_piece_vocabulary(
    raw_train_ds,
    vocabulary_size=VOCAB_SIZE,
    lowercase=True,
    reserved_tokens=["[PAD]", "[UNK]", "[BOS]"],
)

In [5]:
print(f'Длина первой строки из датасета: {len(text[0])}\n')
print(f'Длина второй строки из датасета: {len(text[1])}\n')
print(f'Длина третьей строки из датасета: {len(text[2])}\n')
print(f'Длина словаря: {len(vocab)}\n')
print(f'Последние 20 токенов в словаре: {vocab[-20:]}')

Длина первой строки из датасета: 77

Длина второй строки из датасета: 2000

Длина третьей строки из датасета: 2000

Длина словаря: 980

Последние 20 токенов в словаре: ['##島', '##数', '##日', '##昔', '##校', '##活', '##減', '##濁', '##点', '##知', '##社', '##私', '##算', '##节', '##話', '##走', '##達', '##馬', '##￼', '##�']


In [6]:
tokenizer = keras_hub.tokenizers.WordPieceTokenizer(
    vocabulary=vocab,
    sequence_length=SEQ_LEN,
    lowercase=True,
)

In [7]:
start_packer = keras_hub.layers.StartEndPacker(
    sequence_length=SEQ_LEN,
    start_value=tokenizer.token_to_id("[BOS]"),
)


def preprocess(inputs):
    outputs = tokenizer(inputs)
    features = start_packer(outputs)
    labels = outputs
    return features, labels

train_ds = raw_train_ds.map(preprocess, num_parallel_calls=tf_data.AUTOTUNE).prefetch(
    tf_data.AUTOTUNE
)
val_ds = raw_val_ds.map(preprocess, num_parallel_calls=tf_data.AUTOTUNE).prefetch(
    tf_data.AUTOTUNE
)

In [8]:
inputs = keras.layers.Input(shape=(None,), dtype="int32")
embedding_layer = keras_hub.layers.TokenAndPositionEmbedding(
    vocabulary_size=VOCAB_SIZE,
    sequence_length=SEQ_LEN,
    embedding_dim=EMBED_DIM,
    mask_zero=True,
)
x = embedding_layer(inputs)
for _ in range(NUM_LAYERS):
    decoder_layer = keras_hub.layers.TransformerDecoder(
        num_heads=NUM_HEADS,
        intermediate_dim=FEED_FORWARD_DIM,
    )
    x = decoder_layer(x)

outputs = keras.layers.Dense(VOCAB_SIZE)(x)
model = keras.Model(inputs=inputs, outputs=outputs)
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
perplexity = keras_hub.metrics.Perplexity(from_logits=True, mask_token_id=0)
model.compile(optimizer="adam", loss=loss_fn, metrics=[perplexity])

In [10]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, None, 256)      │       288,768 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder             │ (None, None, 256)      │       329,085 │
│ (TransformerDecoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_1           │ (None, None, 256)      │       329,085 │
│ (TransformerDecoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, None, 1000)     │       257,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,611,816 (13.78 MB)

 Trainable params: 1,203,938 (4.59 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,407,878 (9.19 MB)

In [9]:
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

Epoch 1/14


/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:934: UserWarning: Layer 'position_embedding' (of type PositionEmbedding) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:934: UserWarning: Layer 'query' (of type EinsumDense) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:934: UserWarning: Layer 'key' (of type EinsumDense) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.

     63/Unknown 18s 121ms/step - loss: 5.4013 - perplexity: 274.9004

/usr/lib/python3.11/contextlib.py:158: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)


63/63 ━━━━━━━━━━━━━━━━━━━━ 25s 228ms/step - loss: 5.3907 - perplexity: 272.3602 - val_loss: 4.0516 - val_perplexity: 57.4921
Epoch 2/14
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 69ms/step - loss: 3.9148 - perplexity: 50.1596 - val_loss: 3.8784 - val_perplexity: 48.3490
Epoch 3/14
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 80ms/step - loss: 3.7337 - perplexity: 41.8402 - val_loss: 3.7175 - val_perplexity: 41.1622
Epoch 4/14
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 73ms/step - loss: 3.5107 - perplexity: 33.4806 - val_loss: 3.5098 - val_perplexity: 33.4416
Epoch 5/14
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 85ms/step - loss: 3.2940 - perplexity: 26.9552 - val_loss: 3.3754 - val_perplexity: 29.2367
Epoch 6/14
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 80ms/step - loss: 3.1366 - perplexity: 23.0271 - val_loss: 3.2831 - val_perplexity: 26.6586
Epoch 7/14
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 69ms/step - loss: 3.0162 - perplexity: 20.4145 - val_loss: 3.2238 - val_perplexity: 25.1228
Epoch 8/14
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 84ms/step - loss: 2.9225 - perplex

In [11]:
prompt_tokens = start_packer(tokenizer([""]))
prompt_tokens

<tf.Tensor: shape=(1, 128), dtype=int32, numpy=
array([[2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]],
      dtype=int32)>

In [12]:
def next(prompt, cache, index):
    logits = model(prompt)[:, index - 1, :]
    hidden_states = None
    return logits, hidden_states, cache

In [13]:
sampler = keras_hub.samplers.GreedySampler()
output_tokens = sampler(
    next=next,
    prompt=prompt_tokens,
    index=1,
)
txt = tokenizer.detokenize(output_tokens)
print(f"Greedy search generated text: \n{txt}\n")

Greedy search generated text: 
['[BOS] пациентка , 54 года , обратилась к врачу - с жалобами на боли в области переменных , передъявляет жалобы на работает словом обратилась , что производящие бескопязываются в связи специализированьные субъективного развести . основные']



In [14]:
sampler = keras_hub.samplers.BeamSampler(num_beams=10)
output_tokens = sampler(
    next=next,
    prompt=prompt_tokens,
    index=1,
)
txt = tokenizer.detokenize(output_tokens)
print(f"Beam search generated text: \n{txt}\n")

Beam search generated text: 
['[BOS] п · о · ручебник по авиации общего назначения ( аон ) процедуры по вс : регистрация  · разрешение на радиостанцию  · слг  · страховкапроцедуры по пилоту : пилотское свидетельство  · влэк   · проверка навыков  · английский язык   · летная книжка процедуры по аэрод']



In [15]:
sampler = keras_hub.samplers.RandomSampler()
output_tokens = sampler(
    next=next,
    prompt=prompt_tokens,
    index=1,
)
txt = tokenizer.detokenize(output_tokens)
print(f"Random search generated text: \n{txt}\n")

Random search generated text: 
['[BOS] система в ухохи заменениях органы в профессии на лицалогии возможной так же сахатей ругистрац периодически - комментальных вызодах это случае и то привотке области . манизами они необольшое девеозано , а не труда позифиля посетку их во вездухнизмов']



In [16]:
sampler = keras_hub.samplers.TopKSampler(k=10)
output_tokens = sampler(
    next=next,
    prompt=prompt_tokens,
    index=1,
)
txt = tokenizer.detokenize(output_tokens)
print(f"Top-K search generated text: \n{txt}\n")

Top-K search generated text: 
['[BOS] слово - это политическое стандартом с житолей жань , находилофофособенно в кинстру , дыхание . конце . в специфиратуры технологии с информационной физической сборованием . формы пследва , такая названия знания , начия имеет и проекуску . сам']



In [17]:
sampler = keras_hub.samplers.TopPSampler(p=0.5)
output_tokens = sampler(
    next=next,
    prompt=prompt_tokens,
    index=1,
)
txt = tokenizer.detokenize(output_tokens)
print(f"Top-P search generated text: \n{txt}\n")

Top-P search generated text: 
['[BOS] самостоятельно связырь между образованием производится в образование страницы и формирования и средством региональные инструкции . включаются образование и заполнительной продуктов предпоставления , редактирования и и проекта . располён , в информаци']



In [18]:
class TopKTextGenerator(keras.callbacks.Callback):
    """A callback to generate text from a trained model using top-k."""

    def __init__(self, k):
        self.sampler = keras_hub.samplers.TopKSampler(k)

    def on_epoch_end(self, epoch, logs=None):
        output_tokens = self.sampler(
            next=next,
            prompt=prompt_tokens,
            index=1,
        )
        txt = tokenizer.detokenize(output_tokens)
        print(f"Top-K search generated text: \n{txt}\n")


text_generation_callback = TopKTextGenerator(k=10)
model.fit(train_ds.take(1), verbose=2, epochs=2, callbacks=[text_generation_callback])

Epoch 1/2
Top-K search generated text: 
['[BOS] обознательное практить процессоры , первышкует название документоллекту : информационной булчевшей предмерческой нагрузке , предрасеционных и контрафота . в карты названия . среднем парамотили , у критеризованию , контенсирования']

1/1 - 8s - 8s/step - loss: 2.5576 - perplexity: 12.9047
Epoch 2/2
Top-K search generated text: 
['[BOS] родчик производится как исходным научного предложен с истоятельств , как проблема представление о первого , первый предприятия , с которых условиях , а также вывода процесс организм слуха . парал , профедерации указаться , что интернетических образом посколь']

1/1 - 8s - 8s/step - loss: 2.4468 - perplexity: 11.5508
